In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer

# 1. Load Data & Basic Feature Engineering
train_data = pd.read_csv('dataset/train.csv')
train_data['family_size'] = train_data['SibSp'] + train_data['Parch']

# 2. FIX: Properly separate Features (X_train) and Target (y_train)
y_train = train_data["Survived"].copy()
# Drop the target and the junk columns all at once
X_train = train_data.drop(["Survived", "Cabin", "Ticket", "Name", "PassengerId"], axis=1) 

# 3. Define Custom Transformer
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state
        
    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  
        
    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
        
    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

# 4. Route the specific columns
num_attribs = ["Pclass", "Age", "SibSp", "Parch", "Fare", "family_size"]
cat_attribs = ["Sex", "Embarked"] # Name is safely removed
cluster_attribs = ["Age", "Fare"]

# 5. FIX: Correct the Imputers in the sub-pipelines
nums_pipeline = make_pipeline(
    IterativeImputer(initial_strategy="median", random_state=42), 
    StandardScaler()
)

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"), 
    OneHotEncoder(handle_unknown="ignore")
)

clusters_pipeline = make_pipeline(
    IterativeImputer(initial_strategy="median", random_state=42),
    StandardScaler(),
    ClusterSimilarity(n_clusters=5, gamma=1.0, random_state=42)
)

# 6. FIX: Integrate the custom cluster pipeline
preprocessing = ColumnTransformer([
    ("num", nums_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
    ("cluster", clusters_pipeline, cluster_attribs)
])

# 7. FIX: Execute the pipeline to create the final mathematically ready dataset
X_train_prepared = preprocessing.fit_transform(X_train)

# Print the shape to confirm it worked (Rows, Columns)
print("Final Dataset Shape:", X_train_prepared.shape)
X_train.head()

Final Dataset Shape: (891, 16)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,family_size
0,3,male,22.0,1,0,7.2500,S,1
1,1,female,38.0,1,0,71.2833,C,1
2,3,female,26.0,0,0,7.9250,S,0
3,1,female,35.0,1,0,53.1000,S,1
4,3,male,35.0,0,0,8.0500,S,0
